# Projeto de Equações Diferenciais: método de Euler aplicado à descida do gradiente

Este caderno reúne a parte computacional do trabalho e acompanha o raciocínio do relatório. Ele está dividido em quatro partes. Na primeira, testamos o método de Euler sobre uma equação diferencial cuja solução exata é conhecida. Na segunda, testamos a descida do gradiente separadamente, sem recorrer ao método de Euler. Na terceira, colocamos os resultados das duas partes lado a lado e mostramos que são iguais. Na quarta, aplicamos a descida do gradiente a um problema de otimização real, o ajuste de uma reta a dados por regressão linear. As células devem ser executadas na ordem em que aparecem, de cima para baixo.

## Preparação

Começamos importando as bibliotecas e fixando os valores que serão usados nas três primeiras partes. O caso modelo é a função de erro quadrática $L(\theta) = \frac{a}{2}\theta^2$, cujo gradiente é $\nabla L(\theta) = a\theta$. Adotamos $a = 10$ e partimos de $\theta_0 = 1$. Escolhemos seis tamanhos de passo, de 0,01 a 0,25, para observar as diferentes regiões de comportamento em relação ao valor $2/a = 0{,}2$, que, como veremos, é o limite de estabilidade do método.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

A = 10.0                                  # curvatura da parábola (o "a" do texto)
THETA0 = 1.0                              # ponto de partida
PASSOS = [0.01, 0.05, 0.1, 0.15, 0.2, 0.25]   # os seis passos do relatório
T = 2.0                                   # tempo total da simulação
# uma cor para cada passo, gerada automaticamente
CORES = dict(zip(PASSOS, plt.cm.viridis(np.linspace(0.05, 0.95, len(PASSOS)))))

## Parte 1: o método de Euler

A descida do gradiente pode ser vista como o movimento de um parâmetro que desce continuamente a ladeira do erro, o que é descrito pela equação diferencial

$$\frac{d\theta}{dt} = -\nabla L(\theta) = -a\theta,$$

conhecida como fluxo de gradiente. Para esse caso modelo, a solução exata é $\theta(t) = \theta_0 e^{-at}$, que usaremos como referência. O método de Euler aproxima essa solução em passos de tamanho $h$, seguindo a regra

$$\theta_{n+1} = \theta_n + h\, f(\theta_n), \qquad f(\theta) = -\nabla L(\theta).$$

A célula a seguir define o gradiente, o lado direito da equação diferencial, a solução exata e uma função que aplica o método de Euler a qualquer equação, guardando todos os valores obtidos.

In [ ]:
def gradiente(theta):
    """Gradiente de L(θ) = (a/2)θ²: é a inclinação da ladeira."""
    return A * theta

def f(theta):
    """Lado direito da EDO do fluxo de gradiente: dθ/dt = -∇L(θ)."""
    return -gradiente(theta)

def exata(t):
    """Solução exata da EDO: θ(t) = θ0 · e^(-a t)."""
    return THETA0 * np.exp(-A * t)

def euler(f, y0, h, n_passos):
    """Método de Euler: y_novo = y_antigo + h · f(y_antigo). Guarda todos os valores."""
    y = np.asarray(y0, dtype=float)
    valores = [y]
    for _ in range(n_passos):
        y = y + h * f(y)
        valores.append(y)
    return np.array(valores)

Agora simulamos o método de Euler até $t = 2$ com os seis passos e desenhamos as trajetórias junto com a solução exata. Esperamos ver comportamentos bem diferentes: com passos pequenos, a curva deve acompanhar a solução exata; com passos intermediários, deve oscilar em torno de zero e ainda assim convergir; com passos grandes, deve se afastar cada vez mais de zero. O passo $h = 0{,}1$ leva o método ao mínimo em um único passo. O eixo vertical foi cortado em $\pm 2$ para que a curva que diverge não esconda as demais.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
t_fino = np.linspace(0, T, 400)
ax.plot(t_fino, exata(t_fino), "k", lw=2, label="solução exata")

resultados_euler = {}
for h in PASSOS:
    n = int(round(T / h))
    t = np.arange(n + 1) * h
    resultados_euler[h] = euler(f, THETA0, h, n)
    ax.plot(t, resultados_euler[h], "o-", color=CORES[h], ms=4, label=f"Euler, h = {h}")

ax.set_ylim(-2, 2)
ax.axhline(0, color="gray", lw=0.5)
ax.set_xlabel("tempo t")
ax.set_ylabel("θ")
ax.set_title("Parte 1: método de Euler contra a solução exata")
ax.legend()
plt.show()

O gráfico confirma o que esperávamos. A cada passo, o método de Euler multiplica $\theta$ pelo fator $(1 - ha)$. Esse fator vale $0{,}9$ para $h = 0{,}01$, $0{,}5$ para $h = 0{,}05$, $0$ para $h = 0{,}1$, $-0{,}5$ para $h = 0{,}15$, $-1$ para $h = 0{,}2$ e $-1{,}5$ para $h = 0{,}25$. Quando o módulo do fator é menor que 1, $\theta$ tende a zero, e o sinal negativo explica a oscilação. Com $h = 0{,}2$, o fator é exatamente $-1$, e os valores alternam sem decair. Quando o módulo passa de 1, os valores crescem a cada passo. Isso leva à condição de estabilidade $h < 2/a$.

## Parte 2: a descida do gradiente

Agora esquecemos, por um momento, que existe uma equação diferencial. A descida do gradiente é um algoritmo de otimização: a cada iteração, o parâmetro é atualizado no sentido oposto ao do gradiente,

$$\theta_{n+1} = \theta_n - \eta\,\nabla L(\theta_n),$$

em que $\eta$ é a taxa de aprendizado. A função abaixo foi escrita de forma independente, sem chamar a função de Euler. Isso é importante: se uma usasse a outra, a comparação da próxima parte não provaria nada.

In [ ]:
def descida_gradiente(grad, theta0, taxa, n_passos):
    """Descida do gradiente: θ_novo = θ_antigo - taxa · ∇L(θ_antigo). Não usa o Euler."""
    theta = np.asarray(theta0, dtype=float)
    historico = [theta]
    for _ in range(n_passos):
        theta = theta - taxa * grad(theta)
        historico.append(theta)
    return np.array(historico)

Rodamos a descida do gradiente com o mesmo ponto de partida e com taxas de aprendizado iguais aos passos usados na parte 1.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(t_fino, exata(t_fino), "k", lw=2, label="solução exata")

resultados_gd = {}
for h in PASSOS:
    n = int(round(T / h))
    t = np.arange(n + 1) * h
    resultados_gd[h] = descida_gradiente(gradiente, THETA0, h, n)
    ax.plot(t, resultados_gd[h], "o-", color=CORES[h], ms=4, label=f"descida do gradiente, taxa = {h}")

ax.set_ylim(-2, 2)
ax.axhline(0, color="gray", lw=0.5)
ax.set_xlabel("iterações × taxa de aprendizado")
ax.set_ylabel("θ")
ax.set_title("Parte 2: descida do gradiente")
ax.legend()
plt.show()

## Parte 3: as duas sequências são iguais

Agora comparamos os resultados da parte 1 e da parte 2. Se a descida do gradiente for mesmo o método de Euler aplicado ao fluxo de gradiente, os valores devem ser os mesmos em todos os passos. A célula abaixo calcula a maior diferença entre as duas sequências, verifica se elas são exatamente iguais e mostra os primeiros valores lado a lado, tanto para o passo pequeno, em que o método converge, quanto para o passo grande, em que ele diverge.

In [ ]:
print(f"{'passo h':>8} {'passos':>7} {'maior diferença':>16} {'são iguais?':>12}")
for h in PASSOS:
    diferenca = np.max(np.abs(resultados_euler[h] - resultados_gd[h]))
    iguais = np.array_equal(resultados_euler[h], resultados_gd[h])
    print(f"{h:>8} {len(resultados_euler[h]) - 1:>7} {diferenca:>16.1e} {str(iguais):>12}")

for h in (0.05, 0.25):
    print(f"\nPrimeiros passos com h = {h}")
    print(f"{'passo':>6} {'Euler':>14} {'Descida do gradiente':>22}")
    for i in range(6):
        print(f"{i:>6} {resultados_euler[h][i]:>14.6f} {resultados_gd[h][i]:>22.6f}")

Por fim, colocamos as duas sequências no mesmo gráfico, com o método de Euler em traço grosso e a descida do gradiente em pontos pretos.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
for h in PASSOS:
    n = len(resultados_euler[h]) - 1
    t = np.arange(n + 1) * h
    ax.plot(t, resultados_euler[h], "-", color=CORES[h], lw=4, alpha=0.5, label=f"Euler, h = {h}")
    ax.plot(t, resultados_gd[h], "k.", ms=5)
ax.plot([], [], "k.", label="descida do gradiente (mesmo valor)")
ax.set_ylim(-2, 2)
ax.axhline(0, color="gray", lw=0.5)
ax.set_xlabel("tempo t")
ax.set_ylabel("θ")
ax.set_title("Parte 3: as duas sequências coincidem")
ax.legend()
plt.show()

A diferença é exatamente zero em todos os casos, inclusive quando o método diverge, e os pontos pretos caem sobre as curvas coloridas. Isso não é coincidência. Com $f = -\nabla L$, o passo de Euler é $\theta_n + h\,(-\nabla L(\theta_n))$, que é a mesma operação que $\theta_n - \eta\,\nabla L(\theta_n)$ com $\eta = h$. A taxa de aprendizado é, portanto, o passo do método de Euler, e tudo o que vale para o erro e para a estabilidade de um vale também para o outro.

## Parte 4: a otimização na prática

Até aqui trabalhamos com uma função de uma variável. Vamos agora ver a descida do gradiente resolvendo um problema de otimização de verdade: o ajuste de uma reta a um conjunto de pontos. Otimizar, nesse caso, significa escolher os parâmetros da reta, a inclinação $w$ e o ponto $b$ em que ela cruza o eixo vertical, de modo a minimizar uma função de erro, o erro quadrático médio

$$L(w, b) = \frac{1}{N}\sum_{i=1}^{N}\big(w x_i + b - y_i\big)^2.$$

Geramos os dados em torno da reta $y = 2x + 1$, somando um ruído aleatório, de modo que sabemos de antemão qual é a resposta esperada. A semente do gerador de números aleatórios é fixa, para que os resultados sejam os mesmos a cada execução.

In [ ]:
rng = np.random.default_rng(0)
N = 50
x = rng.uniform(0, 1, N)
y = 2.0 * x + 1.0 + rng.normal(0, 0.1, N)

fig, ax = plt.subplots(figsize=(6, 4))
ax.scatter(x, y, color="#7F77DD")
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_title("Parte 4: os dados (reta verdadeira y = 2x + 1, com ruído)")
plt.show()

As funções abaixo descrevem o problema. A primeira calcula as previsões da reta, a segunda mede o erro e a terceira calcula o gradiente do erro em relação a $w$ e $b$, que é

$$\frac{\partial L}{\partial w} = \frac{2}{N}\sum_{i=1}^{N}\big(w x_i + b - y_i\big)x_i, \qquad \frac{\partial L}{\partial b} = \frac{2}{N}\sum_{i=1}^{N}\big(w x_i + b - y_i\big).$$

In [ ]:
def previsao(p):
    """Reta com parâmetros p = [inclinação w, ponto de corte b]."""
    w, b = p
    return w * x + b

def erro_reg(p):
    """Erro quadrático médio entre a reta e os pontos."""
    return np.mean((previsao(p) - y) ** 2)

def gradiente_reg(p):
    """Gradiente do erro em relação a (w, b)."""
    residuo = previsao(p) - y
    return np.array([2 * np.mean(residuo * x), 2 * np.mean(residuo)])

Antes de escolher a taxa de aprendizado, calculamos o limite de estabilidade. Com dois parâmetros, a curvatura da função de erro é descrita pela matriz Hessiana, e o papel do número $a$ da parte 1 passa a ser desempenhado pelo seu maior autovalor, $\lambda_{\max}$. Por analogia com o caso de uma variável, esperamos que o método seja estável para $h < 2/\lambda_{\max}$. Essa condição é dedução própria, e não um resultado retirado de uma fonte.

In [ ]:
H_reg = 2 * np.array([[np.mean(x**2), np.mean(x)],
                      [np.mean(x),    1.0       ]])
autovalores = np.linalg.eigvalsh(H_reg)
limite = 2 / autovalores.max()
print("Curvaturas (autovalores da Hessiana):", np.round(autovalores, 3))
print(f"Limite de estabilidade 2/λ_max = {limite:.3f}")

Agora rodamos a descida do gradiente a partir da reta $w = 0$, $b = 0$, com três taxas de aprendizado: uma pequena, 0,05; uma boa, 0,5; e uma acima do limite calculado, 0,85. Em cada caso damos 200 passos e imprimimos os valores finais.

In [ ]:
p0 = np.array([0.0, 0.0])
n_passos = 200
taxas = {"pequena": 0.05, "boa": 0.5, "grande demais": 0.85}

historicos = {nome: descida_gradiente(gradiente_reg, p0, taxa, n_passos)
              for nome, taxa in taxas.items()}

for nome, taxa in taxas.items():
    p_final = historicos[nome][-1]
    print(f"taxa {nome:>13} ({taxa}): w = {p_final[0]:.4g}, b = {p_final[1]:.4g}, erro = {erro_reg(p_final):.4g}")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
for nome, taxa in taxas.items():
    erros = [erro_reg(p) for p in historicos[nome]]
    ax.plot(erros, label=f"taxa {nome} ({taxa})")
ax.set_yscale("log")
ax.set_ylim(1e-3, 1e3)   # corta o eixo para a curva que explode não esmagar as outras
ax.set_xlabel("passo")
ax.set_ylabel("erro (escala logarítmica)")
ax.set_title("O erro ao longo da descida, para três taxas de aprendizado")
ax.legend()
plt.show()

Os resultados mostram os três comportamentos que a parte 1 já havia previsto. Com a taxa pequena, o erro cai, mas muito devagar, e ao fim dos 200 passos a reta ainda não chegou à resposta. Com a taxa boa, o erro cai rapidamente e se estabiliza perto do seu valor mínimo. Com a taxa acima do limite, o erro cresce a cada passo e o método diverge, exatamente como o passo grande da parte 1. O eixo vertical, em escala logarítmica, foi cortado para que a curva que diverge não esconda as outras.

Para enxergar a otimização acontecendo, desenhamos a reta obtida com a taxa boa em alguns passos escolhidos, junto com os dados. A reta começa em $y = 0$, no passo 0, e vai se aproximando dos pontos.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5))
ax.scatter(x, y, color="gray", alpha=0.6, label="dados")
xs = np.linspace(0, 1, 50)
for passo in [0, 2, 10, 50, 200]:
    w, b = historicos["boa"][passo]
    ax.plot(xs, w * xs + b, label=f"passo {passo}")
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_title("A reta se ajustando aos dados (taxa boa)")
ax.legend()
plt.show()

Como a regressão linear tem solução exata conhecida, o método dos mínimos quadrados, podemos conferir o resultado da descida do gradiente. Os dois valores coincidem até a quarta casa decimal, e ambos ficam próximos da reta usada para gerar os dados, $y = 2x + 1$, com a pequena diferença causada pelo ruído.

In [ ]:
w_otimo, b_otimo = np.polyfit(x, y, 1)
w_gd, b_gd = historicos["boa"][-1]
print(f"Descida do gradiente: w = {w_gd:.5f}, b = {b_gd:.5f}")
print(f"Mínimos quadrados   : w = {w_otimo:.5f}, b = {b_otimo:.5f}")

## Juntando as partes 3 e 4

Falta verificar que a igualdade entre o método de Euler e a descida do gradiente, mostrada na parte 3, continua valendo para o problema de regressão. Escrevemos a equação diferencial do fluxo de gradiente para os dois parâmetros e aplicamos o método de Euler com o mesmo passo da taxa boa.

In [ ]:
def f_reg(p):
    """EDO do fluxo de gradiente para a regressão: dp/dt = -∇L(p)."""
    return -gradiente_reg(p)

taxa = 0.5
passos_euler = euler(f_reg, p0, taxa, n_passos)
passos_gd = descida_gradiente(gradiente_reg, p0, taxa, n_passos)
print("Maior diferença entre Euler e descida do gradiente:", np.max(np.abs(passos_euler - passos_gd)))
print("São exatamente iguais?", np.array_equal(passos_euler, passos_gd))

Por fim, resolvemos a mesma equação diferencial com um método numérico de alta precisão, que faz o papel de solução exata, e medimos o quanto a descida do gradiente se afasta dela para vários tamanhos de passo.

In [ ]:
from scipy.integrate import solve_ivp

T_reg = 10.0
print(f"{'passo h':>8} {'maior diferença para a EDO exata':>34}")
for h in [0.4, 0.2, 0.1, 0.05]:
    n = int(round(T_reg / h))
    t_grade = np.arange(n + 1) * h
    gd = descida_gradiente(gradiente_reg, p0, h, n)
    sol = solve_ivp(lambda t, p: f_reg(p), [0, T_reg], p0, t_eval=t_grade, rtol=1e-10, atol=1e-12)
    print(f"{h:>8} {np.max(np.abs(gd - sol.y.T)):>34.2e}")

A diferença para a solução precisa diminui quando o passo diminui, com razões entre erros sucessivos que se aproximam de 2, o que é o comportamento de primeira ordem esperado para o método de Euler. Isso fecha o raciocínio do trabalho: a descida do gradiente é o método de Euler aplicado ao fluxo de gradiente, isso vale tanto para uma parábola quanto para uma regressão linear, e as propriedades de erro e de estabilidade do método de Euler explicam o comportamento da taxa de aprendizado na otimização.